In [1]:
import math, torch, torch.nn as nn, torch.nn.functional as F

In [2]:
# Inputs
KMIN, KMAX = 2.0, 200.0          # kappa in [2, 200] -> sd(S) in [0.29, 0.03]
N = 2                            # number of bidders (game-wide)
DEVICE = torch.device(
    torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
)

In [3]:
# defining our NN: 2 hidden layers of 10 nodes each, using LeCun normalisation for SELU activation functions
# 2 parts: the beta policy (the structure and initial state), and the forward function (how information will move in the NN)
class BetaPolicy(nn.Module):
    def __init__(self, h=10):
        super().__init__()
        self.fc1, self.fc2, self.head = nn.Linear(2, h), nn.Linear(h, h), nn.Linear(h, 2)
        for m in (self.fc1, self.fc2):                      
            nn.init.normal_(m.weight, 0.0, 1.0 / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)

    def forward(self, obs):                                   
        z = F.selu(self.fc2(F.selu(self.fc1(obs))))
        z = self.head(z)
        mu    = torch.sigmoid(z[..., 0])
        kappa = KMIN + (KMAX - KMIN) * torch.sigmoid(z[..., 1])
        return mu, kappa  

In [4]:
class FixedShare:
    #Test double for BetaPolicy: always bids b = c * v.
    def __init__(self, c, kappa=200.0):
        self.c, self.kappa = c, kappa

    def __call__(self, obs):
        shape = obs.shape[:-1]                      # strip the 2-feature axis
        return (torch.full(shape, self.c,     device=obs.device),
                torch.full(shape, self.kappa, device=obs.device))

In [5]:
# Instantiating the network and setting an optimisation method
policy = BetaPolicy().to(DEVICE)
# policy.head.bias.data = torch.logit(torch.tensor([0.2, (5 - KMIN) / (KMAX - KMIN)], device=DEVICE)) # only uncommented for manual bias testing
opt = torch.optim.Adam(policy.parameters(), lr=1e-3)

In [6]:
# defining the auction space, outputing a the share of ones vlaue they bid
def play(bids):                       
    top, _ = bids.max(dim=1, keepdim=True)
    share = (bids == top).float()    
    return share / share.sum(dim=1, keepdim=True).clamp_min(1.0)

def check_play(M=1 << 16):
    v = torch.rand(M, N, device=DEVICE); c = (N - 1) / N
    bids = c * v
    rev = (play(bids) * bids).sum(1).mean()
    print(rev.item(), (N - 1) / (N + 1))   # comparing calculated result to the analytical result
    
def check_revenue(tol=5e-3, M=1 << 16):
    c, bench = (N - 1) / N, (N - 1) / (N + 1)
    rule = FixedShare(c)

    err = max(abs(revenue_mc(rule, M=M) - bench),
              abs(revenue_cdf(rule)     - bench),
              abs(efficiencyR(rule, M=M) - c))

    print(f"revenue err {err:.1e}  tol {tol:.0e}  {'ok' if err <= tol else 'FAIL'}")
    assert err <= tol, "revenue validation failed -- check play(), sample_bids(), field_cdf()"
    # print(revenue_mc(rule, N, M=M), revenue_cdf(rule, N), efficiencyR(rule, N, M=M))   # uncomment if revenue check fails

In [7]:
# defining the NN input 
def obs(v):  return torch.stack([v, torch.full_like(v, 1.0 / N)], -1)

# defines a function that runs the NN outputing bid share
def sample_bids(policy, v):
    mu, kappa = policy(obs(v))
    a, b = mu * kappa, (1 - mu) * kappa
    S = torch.distributions.Beta(a, b).rsample()     
    return v * S, S, mu, kappa

In [8]:
# defining athe self-play loop 
def selfplay_step(policy, opt, M=1 << 15):
    v = torch.rand(M, N, device=DEVICE)
    bids, S, mu, kappa = sample_bids(policy, v)
    a = (mu * kappa).clamp_min(1.0)                   
    b = ((1 - mu) * kappa).clamp_min(1.0)   # min of 1 keeps the log_prob finite
    logp = torch.distributions.Beta(a, b).log_prob(S)

    payoff = play(bids) * (v - bids)                  
    adv = (payoff - payoff.mean()).detach()         
    loss = -(logp * adv).mean()

    opt.zero_grad(); loss.backward(); opt.step()
    return payoff.mean().item(), mu.mean().item(), kappa.mean().item()

In [9]:
# building an empirical CDF of rival bids
@torch.no_grad()
def field_cdf(policy, K=256, n=1 << 16):
    b, _, _, _ = sample_bids(policy, torch.rand(n, device=DEVICE))
    grid = torch.cat([b.new_zeros(1), torch.quantile(b, torch.linspace(0, 1, K, device=DEVICE)), b.new_ones(1)])
    qs   = torch.cat([b.new_zeros(1), torch.linspace(0, 1, K, device=DEVICE), b.new_ones(1)])
    return grid, qs

# perform linear interpolation for a given bid
def interp_F(x, grid, qs):
    x = x.clamp(grid[0], grid[-1] - 1e-9)
    i = torch.searchsorted(grid, x).clamp(1, len(grid) - 1)
    w = (x - grid[i - 1]) / (grid[i] - grid[i - 1]).clamp_min(1e-12)
    return (qs[i - 1] + w * (qs[i] - qs[i - 1])).clamp(1e-6, 1 - 1e-6)

# optimises toward the analytic solution
def pseudo_step(policy, opt, M=4096, h=1e-3):
    grid, qs = field_cdf(policy)
    v = torch.rand(M, N, device=DEVICE)
    bid, _, mu, kappa = sample_bids(policy, v)

    Fb = interp_F(bid, grid, qs)
    fb = (interp_F(bid + h, grid, qs) - interp_F(bid - h, grid, qs)) / (2 * h)
    win, d_win = Fb ** (N - 1), (N - 1) * Fb ** (N - 2) * fb
    d_pay = d_win * (v - bid) - win
    
    eu        = (win * (v - bid)).mean().item()
    mu_bar    = mu.mean().item()
    kappa_bar = kappa.mean().item()                  

    loss = -(d_pay.detach() * bid).mean()            
    opt.zero_grad(); loss.backward(); opt.step()
    return eu, mu_bar, kappa_bar

In [10]:
# The exploitability measure
@torch.no_grad()
def exploitability(policy, T=101):
    grid, qs = field_cdf(policy)
    v = torch.linspace(0.01, 1, T, device=DEVICE)
    cand = grid.unsqueeze(0).expand(T, -1)
    eu = (interp_F(cand, grid, qs) ** (N - 1) * (v.unsqueeze(1) - cand)).clamp_min(0)
    br = eu.max(dim=1).values
    mu, _ = policy(obs(v)); played = v * mu
    eu_played = interp_F(played, grid, qs) ** (N - 1) * (v - played)
    return ((br - eu_played).mean() / eu_played.mean()).item()

In [11]:
# The expected revenue measure: monte-carlo method
@torch.no_grad()
def revenue_mc(policy, M=1 << 16):
    v = torch.rand(M, N, device=DEVICE)
    bids, _, _, _ = sample_bids(policy, v)
    return (play(bids) * bids).sum(dim=1).mean().item()

# The expected revenue measure: analytic
@torch.no_grad()
def revenue_cdf(policy, K=512, n=1 << 16):
    grid, qs = field_cdf(policy, K=K, n=n)
    b_mid = 0.5 * (grid[1:] + grid[:-1])
    F_mid = 0.5 * (qs[1:] + qs[:-1])
    dF    = (qs[1:] - qs[:-1]).clamp_min(0.0)
    return (b_mid * N * F_mid ** (N - 1) * dF).sum().item()

In [12]:
# Revennue efficiency measure
def efficiencyR(policy, **kw):
    return revenue_mc(policy, **kw) / (N / (N + 1))

# Allocative efficiency measure
# def efficiencyA()

In [13]:
seed = 0
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [14]:
# Pre-run valdations
check_play()
check_revenue()

0.3335381746292114 0.3333333333333333
revenue err 1.6e-03  tol 5e-03  ok


In [15]:
for step in range(1, 2001):
    # payoff, mu, kappa = selfplay_step(policy, opt)
    payoff, mu, kappa = pseudo_step(policy, opt)    # <-- alternative update rule (do not mix in one step)
    if step % 200 == 0:
        print(f"step {step:5d} | payoff {payoff:.4f} | mu {mu:.4f} | kappa {kappa:6.2f}")
print(f"exploitability: {exploitability(policy):.4f}")
print(f"Expected revenue: {revenue_mc(policy):.4f}")
print(f"Revenue Efficiency: {efficiencyR(policy):.4f}")

step   200 | payoff 0.1673 | mu 0.5121 | kappa 175.75
step   400 | payoff 0.1571 | mu 0.5378 | kappa 193.47
step   600 | payoff 0.1610 | mu 0.5353 | kappa 196.90
step   800 | payoff 0.1578 | mu 0.5349 | kappa 198.00
step  1000 | payoff 0.1644 | mu 0.5233 | kappa 198.47
step  1200 | payoff 0.1665 | mu 0.5103 | kappa 198.63
step  1400 | payoff 0.1642 | mu 0.4999 | kappa 198.71
step  1600 | payoff 0.1691 | mu 0.4914 | kappa 198.78
step  1800 | payoff 0.1655 | mu 0.4969 | kappa 198.78
step  2000 | payoff 0.1688 | mu 0.4907 | kappa 198.84
exploitability: 0.0027
Expected revenue: 0.3288
Revenue Efficiency: 0.4923
